# Signatures

Goals: 

- define a signature mini-language to describe mappings between Hilbert-spaces.
- Should be convertible to Einsum notation when possible
- Should support direct sums and tensor products, as well as arbitrary batching.
- ability to distinguish variable and static input shapes.


Tensor products: 
- via tuples
- direct sums: ?

Example:

- Input is Image of arbitrary shape, output is a monochrome 64x64 image, with arbitrary batching


$$ [..., h, w, c] -> [..., 64, 64] $$

Example: 

Input is a video consisting of 

- temporal image data of fixed size
- temporal audio data of fixed size
- temporal text data (subscripts) of variable size, using word2vec embeddings
- static metadata

$$ [t_image, h, w, c], [t_audio, samples, channels], [t_subtitle, language, content]  $$




In [ ]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional, Union

# ---------- AST definitions ----------


@dataclass
class IdentifierType:
    name: str

    def __repr__(self) -> str:
        return f"Id({self.name})"


# DimType variants


@dataclass
class DimEllipsis:
    def __repr__(self) -> str:
        return "DimEllipsis(...)"


@dataclass
class DimInteger:
    value: int

    def __repr__(self) -> str:
        return f"DimInt({self.value})"


@dataclass
class DimIdentifier:
    ident: IdentifierType

    def __repr__(self) -> str:
        return f"DimId({self.ident.name})"


@dataclass
class DimVarArgs:
    ident: IdentifierType
    kind: str  # "*" or "**"

    def __repr__(self) -> str:
        return f"DimVarArgs({self.kind}{self.ident.name})"


DimType = Union[DimEllipsis, DimInteger, DimIdentifier, DimVarArgs]


@dataclass
class ShapeType:
    dims: list[DimType]

    def __repr__(self) -> str:
        return f"Shape({self.dims})"


@dataclass
class ArgSpec:
    args: list[ArgType]

    def __repr__(self) -> str:
        return f"ArgSpec({self.args})"


@dataclass
class ArgTypeIdentifier:
    ident: IdentifierType

    def __repr__(self) -> str:
        return f"ArgId({self.ident.name})"


@dataclass
class ArgTypeApplied:
    ident: IdentifierType
    argspec: ArgSpec

    def __repr__(self) -> str:
        return f"ArgApplied({self.ident.name}, {self.argspec})"


# ArgType is any of these three:
ArgType = Union[ShapeType, ArgTypeIdentifier, ArgTypeApplied]


# ---------- Tokenizer ----------


@dataclass
class Token:
    kind: str  # "IDENT", "INT", "ELLIPSIS", "STAR", "DSTAR", symbols like "[", "]", "(", ")", ",", "EOF"
    value: str
    pos: int  # character index in original string

    def __repr__(self) -> str:
        return f"Token({self.kind}, {self.value!r}, pos={self.pos})"


def tokenize(source: str) -> list[Token]:
    tokens: list[Token] = []
    i = 0
    n = len(source)

    while i < n:
        c = source[i]

        # Skip whitespace
        if c.isspace():
            i += 1
            continue

        # Multi-character punctuation: ellipsis, double star
        if source.startswith("...", i):
            tokens.append(Token("ELLIPSIS", "...", i))
            i += 3
            continue

        if source.startswith("**", i):
            tokens.append(Token("DSTAR", "**", i))
            i += 2
            continue

        # Single-character punctuation / operators
        if c in "[](),*":
            kind = c
            if c == "*":
                kind = "STAR"
            tokens.append(Token(kind, c, i))
            i += 1
            continue

        # Identifier: [A-Za-z]\w*
        if c.isalpha():
            start = i
            i += 1
            while i < n and (source[i].isalnum() or source[i] == "_"):
                i += 1
            value = source[start:i]
            tokens.append(Token("IDENT", value, start))
            continue

        # Integer literal: \d+
        if c.isdigit():
            start = i
            i += 1
            while i < n and source[i].isdigit():
                i += 1
            value = source[start:i]
            tokens.append(Token("INT", value, start))
            continue

        # If we get here, it's an invalid character
        raise SyntaxError(f"Unexpected character {c!r} at position {i}")

    tokens.append(Token("EOF", "", n))
    return tokens


# ---------- Parser ----------


class Parser:
    def __init__(self, tokens: list[Token]):
        self.tokens = tokens
        self.pos = 0

    # Utility methods

    def current(self) -> Token:
        return self.tokens[self.pos]

    def consume(self, kind: str, value: Optional[str] = None) -> Token:
        tok = self.current()
        if tok.kind != kind and tok.value != kind:
            raise SyntaxError(
                f"Expected {kind!r} at position {tok.pos}, got {tok.kind!r} ({tok.value!r})"
            )
        if value is not None and tok.value != value:
            raise SyntaxError(
                f"Expected {value!r} at position {tok.pos}, got {tok.value!r}"
            )
        self.pos += 1
        return tok

    def match(self, kind: str, value: Optional[str] = None) -> bool:
        tok = self.current()
        if tok.kind == kind or tok.value == kind:
            if value is None or tok.value == value:
                return True
        return False

    # Grammar entry point: parse a full ArgSpec string

    def parse_argspec(self) -> ArgSpec:
        """Parse the grammar starting from ArgSpec and ensure full consumption."""
        result = self._parse_argspec()
        if not self.match("EOF"):
            tok = self.current()
            raise SyntaxError(
                f"Unexpected token {tok.kind} {tok.value!r} at position {tok.pos}, "
                f"expected end of input"
            )
        return result

    # ArgSpec      ::= "[" ArgTypeList? "]"
    # ArgTypeList  ::= ArgType ("," ArgType)*

    def _parse_argspec(self) -> ArgSpec:
        self.consume("[")
        args: list[ArgType] = []
        if not self.match("]"):
            args.append(self._parse_argtype())
            while self.match(","):
                self.consume(",")
                args.append(self._parse_argtype())
        self.consume("]")
        return ArgSpec(args=args)

    # ArgType      ::= ShapeType
    #               | IdentifierType ArgSpec?

    def _parse_argtype(self) -> ArgType:
        tok = self.current()

        # ShapeType starts with "("
        if self.match("("):
            return self._parse_shape_type()

        # Otherwise must start with identifier
        if tok.kind == "IDENT":
            ident = self._parse_identifier()

            # Optional ArgSpec: IdentifierType ArgSpec
            if self.match("["):
                argspec = self._parse_argspec()
                return ArgTypeApplied(ident=ident, argspec=argspec)
            return ArgTypeIdentifier(ident=ident)

        raise SyntaxError(
            f"Expected ArgType at position {tok.pos}, got {tok.kind} {tok.value!r}"
        )

    # ShapeType    ::= "(" DimList? ")"
    # DimList      ::= DimType ("," DimType)*

    def _parse_shape_type(self) -> ShapeType:
        self.consume("(")
        dims: list[DimType] = []
        if not self.match(")"):
            dims.append(self._parse_dim_type())
            while self.match(","):
                self.consume(",")
                dims.append(self._parse_dim_type())
        self.consume(")")
        return ShapeType(dims=dims)

    # DimType      ::= "..."
    #               | IntegerLiteral
    #               | "*"  IdentifierType
    #               | "**" IdentifierType
    #               | IdentifierType

    def _parse_dim_type(self) -> DimType:
        tok = self.current()

        if tok.kind == "ELLIPSIS":
            self.consume("ELLIPSIS")
            return DimEllipsis()

        if tok.kind == "INT":
            self.consume("INT")
            return DimInteger(int(tok.value))

        if tok.kind == "DSTAR":
            self.consume("DSTAR")
            ident = self._parse_identifier()
            return DimVarArgs(ident=ident, kind="**")

        if tok.kind == "STAR":
            self.consume("STAR")
            ident = self._parse_identifier()
            return DimVarArgs(ident=ident, kind="*")

        if tok.kind == "IDENT":
            ident = self._parse_identifier()
            return DimIdentifier(ident=ident)

        raise SyntaxError(
            f"Expected DimType at position {tok.pos}, got {tok.kind} {tok.value!r}"
        )

    # IdentifierType ::= /[A-Za-z]\w*/

    def _parse_identifier(self) -> IdentifierType:
        tok = self.current()
        if tok.kind != "IDENT":
            raise SyntaxError(
                f"Expected identifier at position {tok.pos}, got {tok.kind} {tok.value!r}"
            )
        self.consume("IDENT")
        return IdentifierType(tok.value)


# ---------- Convenience API ----------


def parse_argspec(source: str) -> ArgSpec:
    tokens = tokenize(source)
    parser = Parser(tokens)
    return parser.parse_argspec()


# ---------- Example usage / quick tests ----------

if __name__ == "__main__":
    examples = [
        "[x]",
        "[Tensor[(3, 224, 224)], Label]",
        "[Tensor[(3, ...)], Tensor[(n, m)], Flag]",
        "[Foo[Bar, Baz], (1, 2, n, *rest, **kwrest), ...]",  # last "..." is invalid ArgType -> will error
        "[T1, T2, Tensor[(1, 2, 3)], Tensor[(n, ...)], Tensor[(**k, *x, d)]]",
    ]

    for src in examples:
        print("SOURCE:", src)
        try:
            ast = parse_argspec(src)
            print("AST:   ", ast)
        except SyntaxError as e:
            print("ERROR:", e)
        print("-" * 60)

In [ ]:
from collections.abc import Iterator
from dataclasses import dataclass
from enum import StrEnum
from types import EllipsisType
from typing import Literal, overload

# ---------- AST definitions ----------

type ShapeType = tuple[EllipsisType | int | "Dim", ...]
type ArgType = ShapeType | "IdentifierType" | "GenericType"
type ArgSpec = list[ArgType]


class IdentifierType(str):
    def __new__(cls, name: str) -> IdentifierType:
        # str subclasses must override __new__, not __init__
        obj = super().__new__(cls, name)
        if not obj.isidentifier():
            raise ValueError(f"Invalid identifier: {name}")
        if obj.startswith("_"):
            raise ValueError(f"Identifier cannot start with underscore: {name}")
        return obj


@dataclass(frozen=True, slots=True)
class GenericType:
    ident: IdentifierType
    arg_spec: ArgSpec

    def __str__(self) -> str:
        inner = ", ".join(map(str, self.arg_spec))
        return f"{self.ident}[{inner}]"


class DimKind(StrEnum):
    STATIC = "static"  # plain identifier: n
    DYNAMIC = "dynamic"  # *n
    VARIADIC = "variadic"  # **n


@dataclass(frozen=True, slots=True)
class Dim:
    name: IdentifierType
    kind: DimKind

    def __str__(self) -> str:
        match self.kind:
            case DimKind.STATIC:
                return self.name
            case DimKind.DYNAMIC:
                return f"*{self.name}"
            case DimKind.VARIADIC:
                return f"**{self.name}"
        raise ValueError(f"Unknown dimension kind: {self.kind}")


# ---------- Tokenizer ----------


class TokenKind(StrEnum):
    """Token kinds for the ArgSpec parser."""

    IDENT = "IDENT"
    INT = "INT"
    ELLIPSIS = "..."
    DSTAR = "**"
    STAR = "*"
    LBRACKET = "["
    RBRACKET = "]"
    LPAREN = "("
    RPAREN = ")"
    COMMA = ","
    EOF = "EOF"


@dataclass(frozen=True, slots=True)
class Token:
    pos: int  # character index in original string
    value: str
    kind: TokenKind  # e.g. IDENT, INT, ELLIPSIS, STAR, DSTAR, LBRACKET, ...

    @overload
    def __init__(
        self, pos: int, kind: Literal[TokenKind.IDENT, TokenKind.INT], value: str
    ) -> None: ...
    @overload
    def __init__(
        self,
        pos: int,
        kind: Literal[
            TokenKind.ELLIPSIS,
            TokenKind.STAR,
            TokenKind.DSTAR,
            TokenKind.LBRACKET,
            TokenKind.RBRACKET,
            TokenKind.LPAREN,
            TokenKind.RPAREN,
            TokenKind.COMMA,
            TokenKind.EOF,
        ],
        value: None = None,
    ) -> None: ...
    def __init__(self, pos: int, kind: TokenKind, value: str | None = None) -> None:
        object.__setattr__(self, "pos", int(pos))
        # allow passing either TokenKind or its value (str) if needed
        if not isinstance(kind, TokenKind):
            kind = TokenKind(kind)  # type: ignore[arg-type]
        object.__setattr__(self, "kind", kind)

        if value is None:
            # non-IDENT / non-INT tokens have an implicit value (their lexeme)
            if kind in (TokenKind.IDENT, TokenKind.INT):
                raise AssertionError("IDENT and INT tokens require a value")
            value = kind.value
        elif kind not in (TokenKind.IDENT, TokenKind.INT):
            raise AssertionError("Only IDENT and INT tokens may carry custom values")

        object.__setattr__(self, "value", value)

    def __repr__(self) -> str:
        return f"Token({self.kind.name}, {self.value!r}, pos={self.pos})"


def tokenize(source: str, /) -> Iterator[Token]:
    i = 0
    n = len(source)

    while i < n:
        c = source[i]

        # Skip whitespace
        if c.isspace():
            i += 1
            continue

        # Multi-character punctuation: ellipsis, double star
        if source.startswith("...", i):
            yield Token(i, TokenKind.ELLIPSIS)
            i += 3
            continue

        if source.startswith("**", i):
            yield Token(i, TokenKind.DSTAR)
            i += 2
            continue

        # Single-character punctuation / operators
        if c in "[](),*":
            yield Token(i, TokenKind(c))  # maps "[" -> LBRACKET, etc.
            i += 1
            continue

        # Identifier: [A-Za-z]\w*
        if c.isalpha():
            start = i
            i += 1
            while i < n and (source[i].isalnum() or source[i] == "_"):
                i += 1
            value = source[start:i]
            yield Token(start, TokenKind.IDENT, value)
            continue

        # Integer literal: \d+
        if c.isdigit():
            start = i
            i += 1
            while i < n and source[i].isdigit():
                i += 1
            value = source[start:i]
            yield Token(start, TokenKind.INT, value)
            continue

        # If we get here, it's an invalid character
        raise SyntaxError(f"Unexpected character {c!r} at position {i}")

    yield Token(n, TokenKind.EOF)


# ---------- Parser ----------


class Parser:
    """Recursive-descent parser consuming an Iterator[Token]."""

    def __init__(self, tokens: Iterator[Token]):
        self._tokens = iter(tokens)
        # prime the first token
        try:
            self._current: Token = next(self._tokens)
        except StopIteration:
            # empty stream -> synthetic EOF at pos 0
            self._current = Token(0, TokenKind.EOF)

    @property
    def current(self) -> Token:
        return self._current

    def _advance(self) -> None:
        try:
            self._current = next(self._tokens)
        except StopIteration:
            # Once exhausted, stay on EOF
            if self._current.kind is not TokenKind.EOF:
                self._current = Token(self._current.pos, TokenKind.EOF)

    # Utility methods

    def consume(self, kind: TokenKind) -> Token:
        tok = self.current
        if tok.kind is not kind:
            raise SyntaxError(
                f"Expected {kind.name} at position {tok.pos}, "
                f"got {tok.kind.name} ({tok.value!r})"
            )
        self._advance()
        return tok

    def match(self, kind: TokenKind) -> bool:
        return self.current.kind is kind

    # Grammar entry point: parse a full ArgSpec string

    def parse_argspec(self) -> ArgSpec:
        """Parse the grammar starting from ArgSpec and ensure full consumption."""
        result = self._parse_argspec()
        if self.current.kind is not TokenKind.EOF:
            tok = self.current
            raise SyntaxError(
                f"Unexpected token {tok.kind.name} {tok.value!r} at position {tok.pos}, "
                "expected end of input"
            )
        return result

    # ArgSpec      ::= "[" ArgTypeList? "]"
    # ArgTypeList  ::= ArgType ("," ArgType)*

    def _parse_argspec(self) -> ArgSpec:
        self.consume(TokenKind.LBRACKET)
        args: ArgSpec = []
        if not self.match(TokenKind.RBRACKET):
            args.append(self._parse_argtype())
            while self.match(TokenKind.COMMA):
                self.consume(TokenKind.COMMA)
                args.append(self._parse_argtype())
        self.consume(TokenKind.RBRACKET)
        return args

    # ArgType      ::= ShapeType
    #               | IdentifierType ArgSpec?

    def _parse_argtype(self) -> ArgType:
        tok = self.current

        # ShapeType starts with "("
        if self.match(TokenKind.LPAREN):
            return self._parse_shape_type()

        # Otherwise must start with identifier
        if tok.kind is TokenKind.IDENT:
            ident = self._parse_identifier()

            # Optional ArgSpec: IdentifierType ArgSpec
            if self.match(TokenKind.LBRACKET):
                argspec = self._parse_argspec()
                return GenericType(ident=ident, arg_spec=argspec)
            return ident

        raise SyntaxError(
            f"Expected ArgType at position {tok.pos}, got {tok.kind.name} {tok.value!r}"
        )

    # ShapeType    ::= "(" DimList? ")"
    # DimList      ::= DimType ("," DimType)*

    def _parse_shape_type(self) -> ShapeType:
        self.consume(TokenKind.LPAREN)
        dims: list[EllipsisType | int | Dim] = []
        if not self.match(TokenKind.RPAREN):
            dims.append(self._parse_dim_type())
            while self.match(TokenKind.COMMA):
                self.consume(TokenKind.COMMA)
                dims.append(self._parse_dim_type())
        self.consume(TokenKind.RPAREN)
        return tuple(dims)

    # DimType      ::= "..."
    #               | IntegerLiteral
    #               | IdentifierType
    #               | "*"  IdentifierType
    #               | "**" IdentifierType

    def _parse_dim_type(self) -> EllipsisType | int | Dim:
        tok = self.current

        if tok.kind is TokenKind.ELLIPSIS:
            self.consume(TokenKind.ELLIPSIS)
            return ...

        if tok.kind is TokenKind.INT:
            self.consume(TokenKind.INT)
            return int(tok.value)

        if tok.kind is TokenKind.DSTAR:
            self.consume(TokenKind.DSTAR)
            ident = self._parse_identifier()
            return Dim(name=ident, kind=DimKind.VARIADIC)

        if tok.kind is TokenKind.STAR:
            self.consume(TokenKind.STAR)
            ident = self._parse_identifier()
            return Dim(name=ident, kind=DimKind.DYNAMIC)

        if tok.kind is TokenKind.IDENT:
            ident = self._parse_identifier()
            return Dim(name=ident, kind=DimKind.STATIC)

        raise SyntaxError(
            f"Expected DimType at position {tok.pos}, got {tok.kind.name} {tok.value!r}"
        )

    # IdentifierType ::= /[A-Za-z]\w*/

    def _parse_identifier(self) -> IdentifierType:
        tok = self.current
        if tok.kind is not TokenKind.IDENT:
            raise SyntaxError(
                f"Expected identifier at position {tok.pos}, "
                f"got {tok.kind.name} {tok.value!r}"
            )
        self.consume(TokenKind.IDENT)
        return IdentifierType(tok.value)


# ---------- Convenience API ----------


def parse_argspec(source: str) -> ArgSpec:
    parser = Parser(tokenize(source))
    return parser.parse_argspec()


# ---------- Example usage / quick tests ----------

if __name__ == "__main__":
    examples = [
        "[x]",
        "[Tensor[(3, 224, 224)], Label]",
        "[Tensor[(3, ...)], Tensor[(n, m)], Flag]",
        "[Foo[Bar, Baz], (1, 2, n, *rest, **kwrest), ...]",  # last "..." is invalid ArgType -> will error
        "[T1, T2, Tensor[(1, 2, 3)], Tensor[(n, ...)], Tensor[(**k, *x, d)]]",
    ]

    for src in examples:
        print("SOURCE:", src)
        try:
            ast = parse_argspec(src)
            print("AST:   ", ast)
        except SyntaxError as e:
            print("ERROR:", e)
        print("-" * 60)